# ナンバーリンク 作問ノートブック

## 旧方式 vs 新方式

| | 旧方式 | 新方式 |
|---|---|---|
| 戦略 | ランダム端点配置 → SAT解 | **解先行生成** → 問題を導出 |
| SAT呼び出し | 最大100万回 | **1〜2回** |
| 速度(6x6) | 〜数分 | **< 1秒** |

`NumberLinkGenerator` を使うことで大幅に高速化できる。

In [1]:
from NumberLinkGenerator import NumberLinkGenerator
import time

## 基本的な使い方

In [14]:
# 6x6, 数字5色, 唯一解のみ
t0 = time.time()

gen = NumberLinkGenerator(
    rows=8,
    cols=8,
    num_colors=9,          # 数字(色)の数を固定
    require_unique=True,   # 唯一解のみ採用
    min_path_len=2,        # パスの最小長
    max_attempts=1000000,
)
result = gen.generate()

print(f"生成時間: {time.time() - t0:.3f}秒")

if result:
    print("\n問題グリッド (0=空白):")
    for row in result["puzzle"]:
        print(row)
    print("\n解グリッド:")
    for row in result["solution"]:
        print(row)
    print("\n各パスの長さ:", [len(p) for p in result["paths"]])

[Generator] OK: 243337回目で生成成功 (パス数=9)
生成時間: 656.043秒

問題グリッド (0=空白):
[0, 4, 2, 0, 0, 0, 9, 0]
[0, 0, 0, 0, 8, 0, 3, 0]
[4, 0, 8, 7, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 2, 0, 0, 9]
[0, 0, 6, 1, 3, 7, 0, 5]
[0, 6, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 5, 0, 0, 0, 0]

解グリッド:
[4, 4, 2, 7, 7, 7, 9, 9]
[4, 2, 2, 7, 8, 7, 3, 9]
[4, 2, 8, 7, 8, 7, 3, 9]
[1, 2, 8, 8, 8, 7, 3, 9]
[1, 2, 2, 2, 2, 7, 3, 9]
[1, 6, 6, 1, 3, 7, 3, 5]
[1, 6, 1, 1, 3, 3, 3, 5]
[1, 1, 1, 5, 5, 5, 5, 5]

各パスの長さ: [10, 9, 9, 4, 7, 3, 10, 6, 6]


## パス数を自動決定 (グリッドサイズに応じて自動調整)

In [11]:
# num_colors=None にすると全マス数の25〜40%の色数を自動生成
gen = NumberLinkGenerator(
    rows=8,
    cols=8,
    num_colors=None,
    require_unique=True,
    max_attempts=500,
)
t0 = time.time()
result = gen.generate()
print(f"生成時間: {time.time() - t0:.3f}秒")

if result:
    print(f"生成パス数: {len(result['paths'])}")
    print("\n問題グリッド:")
    for row in result["puzzle"]:
        print(row)

[Generator] OK: 25回目で生成成功 (パス数=16)
生成時間: 0.221秒
生成パス数: 16

問題グリッド:
[3, 0, 0, 3, 10, 0, 0, 4]
[13, 15, 0, 0, 15, 9, 10, 0]
[0, 0, 0, 0, 13, 0, 4, 0]
[0, 0, 14, 16, 2, 0, 11, 11]
[14, 7, 0, 16, 0, 9, 8, 6]
[12, 5, 0, 7, 0, 1, 0, 0]
[0, 0, 0, 5, 2, 0, 0, 0]
[0, 12, 1, 0, 0, 0, 8, 6]


## パス数を範囲指定

In [16]:
# num_colors=(lo, hi) でパス数の範囲を指定
gen = NumberLinkGenerator(
    rows=6,
    cols=6,
    num_colors=(4, 6),     # 4〜6本
    require_unique=True,
    max_attempts=500,
)
t0 = time.time()
result = gen.generate()
print(f"生成時間: {time.time() - t0:.3f}秒")

if result:
    print(f"生成パス数: {len(result['paths'])}")
    for row in result["puzzle"]:
        print(row)

[Generator] OK: 9回目で生成成功 (パス数=6)
生成時間: 0.057秒
生成パス数: 6
[0, 0, 0, 2, 0, 1]
[0, 0, 0, 4, 0, 0]
[3, 0, 4, 0, 0, 0]
[0, 0, 0, 0, 5, 5]
[0, 3, 2, 0, 0, 0]
[0, 0, 6, 6, 1, 0]


In [ ]:
result["puzzle"]

[[0, 0, 0, 0, 0, 2],
 [1, 0, 4, 3, 0, 0],
 [0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0],
 [5, 4, 3, 0, 1, 0],
 [0, 5, 0, 0, 2, 0]]

## 複数問題をまとめて生成

In [17]:
import time

N = 10  # 生成したい問題数
puzzles = []

t0 = time.time()
for i in range(N):
    gen = NumberLinkGenerator(
        rows=6,
        cols=6,
        num_colors=5,
        require_unique=True,
        max_attempts=500,
    )
    result = gen.generate()
    if result:
        puzzles.append(result)

elapsed = time.time() - t0
print(f"{N}問生成: {len(puzzles)}問成功, 合計 {elapsed:.2f}秒 (1問平均 {elapsed/max(len(puzzles),1):.2f}秒)")

[Generator] OK: 108回目で生成成功 (パス数=5)
[Generator] OK: 153回目で生成成功 (パス数=5)
[Generator] OK: 299回目で生成成功 (パス数=5)
[Generator] OK: 448回目で生成成功 (パス数=5)
[Generator] OK: 313回目で生成成功 (パス数=5)
[Generator] OK: 254回目で生成成功 (パス数=5)
[Generator] OK: 187回目で生成成功 (パス数=5)
[Generator] 500回試行したが生成失敗
[Generator] OK: 329回目で生成成功 (パス数=5)
[Generator] OK: 101回目で生成成功 (パス数=5)
10問生成: 9問成功, 合計 4.21秒 (1問平均 0.47秒)


## NumberLinkSolver との比較 (旧方式 vs 新方式)

In [18]:
import numpy as np
import random
from NumberLinkSolver import NumberLinkSolver

# --- 旧方式 ---
h, w, n, d = 6, 6, 5, 2
r = h * w
arr = [i for i in range(1, n + 1) for _ in range(2)]

print("[旧方式] 開始...")
t0 = time.time()
found_old = -1
for i in range(100000):
    indices = random.sample(range(r), n * 2)
    if any(abs(p1 // w - p2 // w) + abs(p1 % w - p2 % w) <= d for p1, p2 in zip(indices[::2], indices[1::2])):
        continue
    line = np.zeros(r, dtype=int)
    line[indices] = arr
    grid = line.reshape(h, w)
    solution = NumberLinkSolver(grid.tolist()).solve()
    if solution and not any(0 in row for row in solution):
        found_old = i
        break
t_old = time.time() - t0
print(f"  {found_old+1}回目で成功, 時間: {t_old:.2f}秒")

# --- 新方式 ---
print("[新方式] 開始...")
t0 = time.time()
gen = NumberLinkGenerator(rows=6, cols=6, num_colors=5, require_unique=True)
result = gen.generate()
t_new = time.time() - t0
print(f"  時間: {t_new:.2f}秒")

print(f"\n速度比: {t_old/t_new:.1f}倍 高速")

[旧方式] 開始...


KeyboardInterrupt: 